In [1]:
import matplotlib
matplotlib.use('Agg')  # Backend non-interactif

import matplotlib.pyplot as plt
import os
import glob
import numpy as np
import astropy.units as u
import random
import pandas as pd

from astropy.wcs import WCS
from textwrap import fill
from astropy.table import Table, vstack, Column
from astropy.coordinates import SkyCoord
from astropy.coordinates import Angle
from astropy.io import fits
from astropy.coordinates import search_around_sky
from astropy.visualization import simple_norm
from astropy.table import join, hstack

from constantes import LIM_FLUX_CLUSTER, LIM_FLUX_AGN, SEARCH_RADIUS_CLUSTER, SEARCH_RADIUS_AGN, EXT_LIKE_C1, EXT_LIKE_C2, EXT_C1_C2, PNT_DET_ML_SPURIOUS, EXT_LIKE_SPURIOUS, NOMBRE_PHOTONS_MIN
from constantes import print_parameters

print_parameters()

╭───────────────────────────────────────────────╮
│                   PARAMÈTRES                  │
├───────────────────────────────────────────────┤
│ LIM_FLUX_CLUSTER        : 1.00e-15 erg/cm²/s 
│ LIM_FLUX_AGN            : 1.00e-15 erg/cm²/s 
│ SEARCH_RADIUS_CLUSTER   : 20.00    arcsec    
│ SEARCH_RADIUS_AGN       : 10.00    arcsec    
│ EXT_LIKE_C1             : 33       
│ EXT_LIKE_C2             : 15       
│ EXT_C1_C2               : 5        arcsec    
│ EXT_LIKE_C1_new         : 80       
│ EXT_LIKE_C2_new         : 35       
│ EXT_C1_C2_new           : 13       arcsec    
│ window_size             : 2.0      arcmin    
│ MAX_Xamin_PAR_FENESTRON : 2        
│ PNT_DET_ML_SPURIOUS     : 20       
│ EXT_LIKE_SPURIOUS       : 15       
│ NOMBRE_PHOTONS_MIN      : 100      photons   
╰───────────────────────────────────────────────╯


# **Coupe appliquée** 

- Nph_Xamin >= **NOMBRE_PHOTONS_MIN**

- Flux_AGN(input) > **LIM_FLUX_AGN**

- Flux_AMAS(input) > **LIM_FLUX_CLUSTER**

# **Noms des chemins vers les catalogues**

In [2]:
Simulation1 = True
Simulation2 = False
Simulation3 = False

if(Simulation1):
    name_dir = 'Simulation1'
    catalog_path_input_AMAS = os.path.expanduser('~/Documents/TransformerProject/data/Simulation1/XFSII_25_sx_p18_b05rc02_output.csv')
    catalog_path_aftXamin = os.path.expanduser('~/Documents/TransformerProject/data/Simulation1/Xamin_stack_fxabs2e-16/merged_catalog_cleaned.fits')
    catalog_path_AGN      = os.path.expanduser('~/Documents/TransformerProject/data/Simulation1/FS2_MAMBO_AGN.fits')
    path_images = "/local/home/sh275430/Documents/TransformerProject/data/Simulation1/Images-full-40ks/imgs"

if(Simulation2):
    name_dir = 'Simulation2'
    catalog_path_input_AMAS = os.path.expanduser('~/Documents/TransformerProject/data/Simulation2/fsII_25_lensed_AGN1/XFSII_25_p18_b05rc02_lensed_1e13Mo_output_cleaned.fits')
    catalog_path_aftXamin = os.path.expanduser('~/Documents/TransformerProject/data/Simulation2/fsII_25_lensed_AGN1/Xamin_onlyMOSPN/merged_catalog_cleaned.fits')
    catalog_path_AGN      = os.path.expanduser('~/Documents/TransformerProject/data/Simulation2/fsII_25_lensed_AGN1/FS2_MAMBO_AGN_1.fits')
    path_images = "/local/home/sh275430/Documents/TransformerProject/data/Simulation2/fsII_25_lensed_AGN1/imgs"

if(Simulation3):
    name_dir = 'Simulation3'
    catalog_path_input_AMAS = os.path.expanduser('~/Documents/TransformerProject/data/Simulation3/fsII_25_lensed_AGN2/XFSII_25_p18_b05rc02_lensed_1e13Mo_output_cleaned.fits')
    catalog_path_aftXamin = os.path.expanduser('~/Documents/TransformerProject/data/Simulation3/fsII_25_lensed_AGN2/Xamin_onlyMOSPN/merged_catalog_cleaned.fits')
    catalog_path_AGN      = os.path.expanduser('~/Documents/TransformerProject/data/Simulation3/fsII_25_lensed_AGN2/FS2_MAMBO_AGN_2.fits')
    path_images = "/local/home/sh275430/Documents/TransformerProject/data/Simulation3/fsII_25_lensed_AGN2/imgs"

In [3]:
data_Xamin      = Table.read(catalog_path_aftXamin)
data_input_AMAS = Table.read(catalog_path_input_AMAS)
data_input_AGN  = Table.read(catalog_path_AGN)

data_Xamin['new_ID'] = np.arange(len(data_Xamin))
data_Xamin['Ntot'] = data_Xamin['INST0_EXP'] * data_Xamin['PNT_RATE_MOS'] + data_Xamin['INST1_EXP'] * data_Xamin['PNT_RATE_PN']
print(f"\033[1mNombre de sources Xamin:\033[0m\nAvant la coupe sur le nombre de photons: \033[1m{len(data_Xamin)}\033[0m")
data_Xamin = data_Xamin[data_Xamin['Ntot']>=NOMBRE_PHOTONS_MIN]
print(f"Apres la coupe sur le nombre de photons: \033[1m{len(data_Xamin)}\033[0m")
print(f"\nRappel: NOMBRE_PHOTONS_MIN = {NOMBRE_PHOTONS_MIN} photons")


Nombre de sources Xamin:
Avant la coupe sur le nombre de photons: 25319
Apres la coupe sur le nombre de photons: 7316

Rappel: NOMBRE_PHOTONS_MIN = 100 photons


In [4]:
print(min(data_Xamin['Ntot']))
print(max(data_Xamin['Ntot']))

100.02261822088127
119484.0183808482


In [5]:
data_input_AMAS.columns
data_input_AGN.columns

<TableColumns names=('id','ra','dec','z','m','passive','agn_type','Lx_h','Lx_s','Fx_s_abs','Fx_s_G14','NH','Mbh')>

In [6]:
data_numeric = data_Xamin.to_pandas()

# Définition des classes C1 et C2
cond_C1 = np.logical_and((data_numeric['EXT'] > EXT_C1_C2) , (data_numeric['EXT_LIKE'] >= EXT_LIKE_C1))
cond_C2 = np.logical_and(np.logical_and((data_numeric['EXT'] > EXT_C1_C2) , (data_numeric['EXT_LIKE'] < EXT_LIKE_C1)),
                        (data_numeric['EXT_LIKE'] > EXT_LIKE_C2))

n_C1 = sum(cond_C1)
n_C2 = sum(cond_C2)
ni_C1_ni_C2 = len(data_numeric) - (n_C1+n_C2)

print("="*70)
print(f"Catalogue Xamin : {len(data_numeric)} dont Nph >= {NOMBRE_PHOTONS_MIN}")
print(f"\nNombre d'amas dans la classe C1 (EXT>{EXT_C1_C2} ET EXT_LIKE>={EXT_LIKE_C1}): {n_C1}")
print(f"Nombre d'amas dans la classe C2 (EXT>{EXT_C1_C2} ET {EXT_LIKE_C2}<EXT_LIKE<{EXT_LIKE_C1}): {n_C2}")
print(f"Nombre d'amas ni dans C1, ni dans C2: {ni_C1_ni_C2}")
print("="*70)

Catalogue Xamin : 7316 dont Nph >= 100

Nombre d'amas dans la classe C1 (EXT>5 ET EXT_LIKE>=33): 177
Nombre d'amas dans la classe C2 (EXT>5 ET 15<EXT_LIKE<33): 51
Nombre d'amas ni dans C1, ni dans C2: 7088


Remarque: suppresion classique des fausses sources

In [7]:
mask_spurious = np.logical_or(data_Xamin['PNT_DET_ML'] >= PNT_DET_ML_SPURIOUS,data_Xamin['EXT_LIKE'] >= EXT_LIKE_SPURIOUS)
data_Xamin_ClassicCUT = data_Xamin[mask_spurious]
print(f"Nombre de sources apres Xamin: {len(data_Xamin)}")
print(f"Nombre de sources apres Xamin sans les fausses sources: {len(data_Xamin_ClassicCUT)}")

Nombre de sources apres Xamin: 7316
Nombre de sources apres Xamin sans les fausses sources: 7289


- On renomme les colonnes de la table data_befXamin

In [8]:
# Noms actuels des colonnes (pour vérification)
current_cols = data_input_AMAS.colnames
print("Colonnes actuelles:")
print(current_cols)

if(Simulation1):
    new_column_names = ['ID', 'R.A.', 'Dec', 'px', 'yx', 'm200', 'Tsl', 'Lx_soft', 'flux', 'flux_ABS', 'r500', 'z']

    # Renommer uniquement les colonnes 2 à 13
    for i in range(1, 13):
        data_input_AMAS.rename_column(current_cols[i], new_column_names[i-1])

    print("\nColonnes après renommage:")
    print(data_input_AMAS.colnames)

if Simulation2 or Simulation3:
    for colname in data_input_AMAS.colnames:
        # Vérifie si la colonne est numérique (int, float, etc.)
        if data_input_AMAS[colname].dtype.kind in 'iufc':
            # Convertit en float64 (meilleure précision que float32)
            data_input_AMAS[colname] = data_input_AMAS[colname].astype(np.float64)
            
    # Afficher un aperçu de la data_input_AMAS pour vérification
    print(data_input_AMAS.colnames)

Colonnes actuelles:
['3', '513636967', '148.852551198248', '3.66324683120872', '3747.87232503616', '3475.58376807197', '7621999900000', '0.56', '4.07E+41', '3.21E-17', '2.40E-17', '11.52', '0.8423']

Colonnes après renommage:
['3', 'ID', 'R.A.', 'Dec', 'px', 'yx', 'm200', 'Tsl', 'Lx_soft', 'flux', 'flux_ABS', 'r500', 'z']


In [9]:
if(Simulation1):
    #data_input_AGN.rename_column('id', 'object_id')
    data_input_AGN.rename_column('ra', 'ra_mag_gal')
    data_input_AGN.rename_column('dec', 'dec_mag_gal')

# **Filtrage du flux pour les amas et les AGN**

In [10]:
if (Simulation1):
    clefluxAGN = 'Fx_s_abs'
if (Simulation2 or Simulation3):
    clefluxAGN = 'Fx_05_2'

In [11]:
print(f"\n{'='*50}")
print("STATISTIQUES DE FILTRAGE")
print(f"Nombre initial d'amas : {len(data_input_AMAS)}")
print(f"Nombre d'amas après masque (flux_ABS > {LIM_FLUX_CLUSTER}) : {len(data_input_AMAS[data_input_AMAS['flux_ABS'] > LIM_FLUX_CLUSTER])}")
print(f"\n{'='*50}")
print("STATISTIQUES DE FILTRAGE")
print(f"Nombre initial d'AGN : {len(data_input_AGN)}")
print(f"Nombre d'AGN après masque (flux_ABS > {LIM_FLUX_AGN}) : {len(data_input_AGN[data_input_AGN[clefluxAGN] > LIM_FLUX_AGN])}")


mask_flux_cluster = data_input_AMAS['flux_ABS'] > LIM_FLUX_CLUSTER
data_input_AMAS = data_input_AMAS[mask_flux_cluster]

mask_flux_agn = data_input_AGN[clefluxAGN] > LIM_FLUX_AGN
data_input_AGN = data_input_AGN[mask_flux_agn]


STATISTIQUES DE FILTRAGE
Nombre initial d'amas : 18995
Nombre d'amas après masque (flux_ABS > 1e-15) : 2026

STATISTIQUES DE FILTRAGE
Nombre initial d'AGN : 976895
Nombre d'AGN après masque (flux_ABS > 1e-15) : 21273


# **Correlation des catalogues d'entree et celui d'Xamin**

In [12]:
def filter_matched_sources(tableXAMIN, tableINPUT,
                           ra_xamin, dec_xamin,
                           ra_input, dec_input,
                           search_radius):

    search_radius_arcsec = (search_radius* u.deg).to(u.arcsec)  # en arcsec

    # Nettoyage des NaN
    mask_xamin = ~(np.isnan(tableXAMIN[ra_xamin])) & ~(np.isnan(tableXAMIN[dec_xamin]))
    mask_input = ~(np.isnan(tableINPUT[ra_input])) & ~(np.isnan(tableINPUT[dec_input]))
    clean_XAMIN = tableXAMIN[mask_xamin]
    clean_INPUT = tableINPUT[mask_input]

    # Conversion en SkyCoord
    coords_XAMIN = SkyCoord(ra=clean_XAMIN[ra_xamin]*u.deg, dec=clean_XAMIN[dec_xamin]*u.deg)
    coords_INPUT = SkyCoord(ra=clean_INPUT[ra_input]*u.deg, dec=clean_INPUT[dec_input]*u.deg)
    
    # Recherche des paires dans le rayon
    idx_input, idx_xamin, sep2d, _ = search_around_sky(coords_INPUT, coords_XAMIN, seplimit=search_radius_arcsec)
    
    # Récupérer les ID_Xamin correspondants
    matched_xamin_indices = np.unique(idx_xamin)
    matched_ids = clean_XAMIN['ID_Xamin'][matched_xamin_indices]

    # Créer un masque des sources INPUT ayant des correspondances
    matched_mask = np.isin(np.arange(len(tableINPUT)), np.unique(idx_input))
    
    return tableINPUT[matched_mask], np.array(matched_ids)

In [13]:
data_AMAS, list_ID_Xamin_AMAS = filter_matched_sources(data_Xamin, data_input_AMAS,
                                                       ra_xamin = 'EXT_RA', dec_xamin = 'EXT_DEC',
                                                       ra_input = 'R.A.', dec_input = 'Dec',
                                                       search_radius = SEARCH_RADIUS_CLUSTER)

In [14]:
data_AGN, list_ID_Xamin_AGN = filter_matched_sources(data_Xamin, data_input_AGN,
                                                     ra_xamin = 'PNT_RA', dec_xamin = 'PNT_DEC',
                                                     ra_input = 'ra_mag_gal', dec_input = 'dec_mag_gal',
                                                     search_radius = SEARCH_RADIUS_AGN)

In [15]:
separator = "═" * 60

print(f"""
{separator}
        RÉSULTATS DE LA CORRÉLATION DES CATALOGUES
{separator}

\033[94mSources Xamin (Nph ≥ {NOMBRE_PHOTONS_MIN}):\033[0m
   • Total initial : {len(data_Xamin):>6d} sources

\033[91mCorrespondances amas/AGN: \033[0m
{'-'*50}
Amas:
  • Catalogue d'entrée : {len(data_input_AMAS):>6d}
  • Corrélés avec Xamin : \033[91m {len(data_AMAS):>4d} ({len(data_AMAS)/len(data_input_AMAS):.1%}) \033[0m

AGN:
  • Catalogue d'entrée : {len(data_input_AGN):>6d}
  • Corrélés avec Xamin : \033[91m {len(data_AGN):>6d} ({len(data_AGN)/len(data_input_AGN):.1%}) \033[0m

{separator}
""")


════════════════════════════════════════════════════════════
        RÉSULTATS DE LA CORRÉLATION DES CATALOGUES
════════════════════════════════════════════════════════════

Sources Xamin (Nph ≥ 100):
   • Total initial :   7316 sources

Correspondances amas/AGN: 
--------------------------------------------------
Amas:
  • Catalogue d'entrée :   2026
  • Corrélés avec Xamin :   285 (14.1%) 

AGN:
  • Catalogue d'entrée :  21273
  • Corrélés avec Xamin :    7383 (34.7%) 

════════════════════════════════════════════════════════════



In [16]:
data_Xamin[data_Xamin['ID_Xamin']==32378]

BOX_ID_SRC,INST0,BAND0,INST1,BAND1,INST0_EXP,INST1_EXP,INST0_GAPFLAG,INST1_GAPFLAG,GAP_NEIGHBOUR,INPUT_X_IMA,INPUT_Y_IMA,PNT_DET_ML,EXT_DET_ML,EPN_DET_ML,DBL_DET_ML,EPN_ML_EXT,PNT_X_IMA,PNT_X_IMA_ERR,PNT_Y_IMA,PNT_IMA_ERR,PNT_RA,PNT_DEC,PNT_RADEC_ERR,PNT_RATE_MOS,PNT_RATE_MOS_ERR,PNT_RATE_PN,PNT_RATE_PN_ERR,PNT_SCTS_MOS,PNT_SCTS_MOS_ERR,PNT_SCTS_PN,PNT_SCTS_PN_ERR,PNT_BG_MAP_MOS,PNT_BG_MAP_PN,PNT_PIX_DEV,PNT_N_ITER,PNT_CUTRAD,EXT_LIKE,EXT,EXT_ERR,EXT_X_IMA,EXT_X_IMA_ERR,EXT_Y_IMA,EXT_Y_IMA_ERR,EXT_RA,EXT_DEC,EXT_RADEC_ERR,EXT_RATE_MOS,EXT_RATE_MOS_ERR,EXT_RATE_PN,EXT_RATE_PN_ERR,EXT_SCTS_MOS,EXT_SCTS_MOS_ERR,EXT_SCTS_PN,EXT_SCTS_PN_ERR,EXT_BG_MAP_MOS,EXT_BG_MAP_PN,EXT_PIX_DEV,EXT_N_ITER,EXT_CUTRAD,EPN_LIKE_EXT,EPN_LIKE_PNT,EPN_EXT,EPN_EXT_ERR,EPN_RATIO,EPN_RATIO_ERR,EPN_X_IMA,EPN_X_IMA_ERR,EPN_Y_IMA,EPN_Y_IMA_ERR,EPN_RA,EPN_DEC,EPN_RADEC_ERR,EPN_RATE_MOS,EPN_RATE_MOS_ERR,EPN_RATE_PN,EPN_RATE_PN_ERR,EPN_SCTS_MOS,EPN_SCTS_MOS_ERR,EPN_SCTS_PN,EPN_SCTS_PN_ERR,EPN_BG_MAP_MOS,EPN_BG_MAP_PN,EPN_PIX_DEV,EPN_N_ITER,EPN_CUTRAD,DBL_LIKE,DBL_SEP,DBL_SEP_ERR,DBL_RATIO,DBL_RATIO_ERR,DBL_THETA,DBL_X_IMA,DBL_X_IMA_ERR,DBL_Y_IMA,DBL_Y_IMA_ERR,DBL_RA,DBL_DEC,DBL_RADEC_ERR,DBL_RATE_MOS,DBL_RATE_MOS_ERR,DBL_RATE_PN,DBL_RATE_PN_ERR,DBL_SCTS_MOS,DBL_SCTS_MOS_ERR,DBL_SCTS_PN,DBL_SCTS_PN_ERR,DBL_BG_MAP_MOS,DBL_BG_MAP_PN,DBL_PIX_DEV,DBL_N_ITER,DBL_CUTRAD,ID_Xamin,new_ID,Ntot
int32,bytes8,bytes8,bytes8,bytes8,float64,float64,int32,int32,int32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int32,float64,int64,int64,float64


# **Sauvegarde des fichiers**

In [17]:
output_filename = f"AMAS_matches_r{SEARCH_RADIUS_CLUSTER*3600:.0f}arcsec_flux{LIM_FLUX_CLUSTER}_40ks.fits"
output_path = os.path.expanduser(f'/local/home/sh275430/Documents/TransformerSurFenestrons/results/{name_dir}/{output_filename}')

data_AMAS.write(output_path, format='fits', overwrite=True)

with fits.open(output_path, mode='update') as hdul:
    hdr = hdul[1].header
    hdr['COMMENT'] = 'Input AMAS spatially matched source catalog'
    hdr['R_MATCH'] = (SEARCH_RADIUS_CLUSTER*3600, '[arcsec] Matching radius') 
    hdr['N_MATCH'] = (len(data_AMAS), 'Total matched sources')

print(f"Catalogue complet sauvegardé dans : {output_path}")
print(f"Dimensions : {len(data_AMAS)} lignes x {len(data_AMAS.columns)} colonnes")

Catalogue complet sauvegardé dans : /local/home/sh275430/Documents/TransformerSurFenestrons/results/Simulation1/AMAS_matches_r20arcsec_flux1e-15_40ks.fits
Dimensions : 285 lignes x 13 colonnes


In [18]:
output_filename = f"list_ID_Xamin_AMAS_matches_r{SEARCH_RADIUS_CLUSTER*3600:.0f}arcsec_flux{LIM_FLUX_CLUSTER}_40ks.fits"
output_path = os.path.expanduser(f'/local/home/sh275430/Documents/TransformerSurFenestrons/results/{name_dir}/{output_filename}')
np.savetxt(output_path, list_ID_Xamin_AMAS, fmt='%d')

In [19]:
output_filename = f"AGN_matches_r{SEARCH_RADIUS_AGN*3600:.0f}arcsec_flux{LIM_FLUX_AGN}_40ks.fits"
output_path = os.path.expanduser(f'/local/home/sh275430/Documents/TransformerSurFenestrons/results/{name_dir}/{output_filename}')

data_AGN.write(output_path, format='fits', overwrite=True)

with fits.open(output_path, mode='update') as hdul:
    hdr = hdul[1].header
    hdr['COMMENT'] = 'Input AGN spatially matched source catalog'
    hdr['R_MATCH'] = (SEARCH_RADIUS_AGN*3600, '[arcsec] Matching radius') 
    hdr['N_MATCH'] = (len(data_AGN), 'Total matched sources')

print(f"Catalogue complet sauvegardé dans : {output_path}")
print(f"Dimensions : {len(data_AGN)} lignes x {len(data_AGN.columns)} colonnes")

Catalogue complet sauvegardé dans : /local/home/sh275430/Documents/TransformerSurFenestrons/results/Simulation1/AGN_matches_r10arcsec_flux1e-15_40ks.fits
Dimensions : 7383 lignes x 13 colonnes


In [20]:
output_filename = f"list_ID_Xamin_AGN_matches_r{SEARCH_RADIUS_AGN*3600:.0f}arcsec_flux{LIM_FLUX_AGN}_40ks.fits"
output_path = os.path.expanduser(f'/local/home/sh275430/Documents/TransformerSurFenestrons/results/{name_dir}/{output_filename}')
np.savetxt(output_path, list_ID_Xamin_AGN, fmt='%d')

# **Overlays**

In [21]:
list_tile_centers = np.array([[146.75 + d_ra, 1.75 + d_dec] for d_ra in range(5) for d_dec in range(5)])

def trouver_tuile(ra, dec):
    """
    Trouve la tuile dont le centre est le plus proche des coordonnées données.
    Paramètres: ra, dec : Coordonnées en degrés
    Retourne: [ra_centre, dec_centre] : Coordonnées du centre de la tuile la plus proche
    """
    coord = SkyCoord(ra, dec, unit='deg', frame='icrs')
    tile_coords = SkyCoord(list_tile_centers[:,0], list_tile_centers[:,1], unit='deg', frame='icrs')
    distances = coord.separation(tile_coords) # Calcul des distances en une seule opération vectorisée
    i = np.argmin(distances)

    return list_tile_centers[i]

In [22]:
def plot_tverif(id, mr,
                show_Xamin_sources,
                show_SExtractor=False,
                NOMBRE_PHOTONS_NOUVELLE_BORNE=40,
                output_dir=f"/local/home/sh275430/Documents/TransformerSurFenestrons/results/{name_dir}/Overlays",
                imgs_dir=path_images,
                catalog_path_SExtractor=f'~/Documents/TransformerProject/data/{name_dir}/fsII_25_lensed_AGN1/SEX'):
    """
    Plot an image centered on a given source, with axes in arcminutes (ΔRA, ΔDec).
    """

    # --- Coordonnées de la source ---
    ra_Xamin, dec_Xamin = data_Xamin[data_Xamin['ID_Xamin'] == id][['PNT_RA', 'PNT_DEC']][0]
    ra_tile, dec_tile = trouver_tuile(ra_Xamin, dec_Xamin)

    # --- Fichiers ---
    imgs_dir = os.path.expanduser(imgs_dir)
    output_dir = os.path.expanduser(output_dir + ('/Wavelets_images' if mr else '/Photons_images'))
    os.makedirs(output_dir, exist_ok=True)

    img_file = f"Tile-{ra_tile}-{dec_tile}_m12pn_b2_mosaic{'_mr' if mr else ''}.fits"
    img_path = os.path.join(imgs_dir, img_file)

    # --- Lecture image FITS ---
    with fits.open(img_path) as hdul:
        img_data = hdul[0].data
        header = hdul[0].header
        wcs = WCS(header)

    # --- Centre en pixels ---
    center_pix = wcs.world_to_pixel(SkyCoord(ra_Xamin, dec_Xamin, unit='deg'))
    x_center, y_center = center_pix

    # --- Échelle en arcsec/pixel → arcmin ---
    pix_scale = np.abs(wcs.proj_plane_pixel_scales()[0].to(u.arcsec).value)
    pix_scale_arcmin = pix_scale / 60.0

    # --- Rayon = 7 arcmin ---
    taille_image = 2
    radius_pix = int((taille_image * 60) / pix_scale)

    # --- Définir les bords du crop ---
    x_min = max(0, int(np.floor(x_center - radius_pix)))
    x_max = min(img_data.shape[1], int(np.ceil(x_center + radius_pix)))
    y_min = max(0, int(np.floor(y_center - radius_pix)))
    y_max = min(img_data.shape[0], int(np.ceil(y_center + radius_pix)))

    # --- Image crop ---
    cropped_img = img_data[y_min:y_max, x_min:x_max]
    ny, nx = cropped_img.shape

    # Décalage en arcmin de chaque pixel par rapport au centre
    x_extent = (np.array([0, nx]) - (x_center - x_min)) * pix_scale_arcmin
    y_extent = (np.array([0, ny]) - (y_center - y_min)) * pix_scale_arcmin

    # --- Axes arcmin centrés sur (0,0) ---
    x_arcmin = (np.arange(nx) - (x_center - x_min)) * pix_scale_arcmin
    y_arcmin = (np.arange(ny) - (y_center - y_min)) * pix_scale_arcmin

    #print(f"x_min={x_min}, x_max={x_max}, y_min={y_min}, y_max={y_max}")
    #print(f"cropped_img.shape = {cropped_img.shape}")
    
    # --- Figure ---
    fig, ax = plt.subplots(figsize=(12, 10))

    # --- Affichage de l'image ---
    vmin_fixed = 0.
    vmax_fixed = 0.00004
    ax.imshow(
        cropped_img,
        cmap="gray",
        vmin=vmin_fixed,
        vmax=vmax_fixed,
        origin='lower',
        extent=[x_extent[0], x_extent[1], y_extent[0], y_extent[1]]
    )

    # --- Fonction pour filtrer et afficher les sources dans le champ ---
    def plot_sources_arcmin(coords, marker, color, label, cross=False):
        delta_ra = (coords.ra.deg - ra_Xamin) * 60 * np.cos(np.deg2rad(dec_Xamin))
        delta_dec = (coords.dec.deg - dec_Xamin) * 60

        mask = (
            (delta_ra >= x_arcmin[0]) & (delta_ra <= x_arcmin[-1]) &
            (delta_dec >= y_arcmin[0]) & (delta_dec <= y_arcmin[-1])
        )

        if cross:
            ax.scatter(delta_ra[mask], delta_dec[mask],
                       marker=marker, s=550, color=color, linewidths=2, label=label)
        else:
            ax.scatter(delta_ra[mask], delta_dec[mask],
                       marker=marker, s=200, facecolors='none',
                       edgecolors=color, linewidths=2, label=label)

    # --- Affichage des sources ---

    coords_clusters = SkyCoord(data_input_AMAS["R.A."], data_input_AMAS["Dec"], unit='deg')
    plot_sources_arcmin(coords_clusters, 's', 'cyan', "Amas sans correlation avec Xamin")

    coords_clusters = SkyCoord(data_AMAS["R.A."], data_AMAS["Dec"], unit='deg')
    plot_sources_arcmin(coords_clusters, 's', 'blue', "Amas")

    coords_agn = SkyCoord(data_input_AGN["ra_mag_gal"], data_input_AGN["dec_mag_gal"], unit='deg')
    plot_sources_arcmin(coords_agn, '*', 'cyan', "AGN sans correlation avec Xamin")

    coords_agn = SkyCoord(data_AGN["ra_mag_gal"], data_AGN["dec_mag_gal"], unit='deg')
    plot_sources_arcmin(coords_agn, '*', 'blue', "AGN")

    if show_Xamin_sources:
        coords_xamin = SkyCoord(data_Xamin["PNT_RA"], data_Xamin["PNT_DEC"], unit='deg')
        plot_sources_arcmin(coords_xamin, 's', 'limegreen', f"Xamin > {NOMBRE_PHOTONS_MIN}ph")

        data_Xamin['Ntot'] = 2 * data_Xamin['PNT_SCTS_MOS'] + data_Xamin['PNT_SCTS_PN']
        sources_photons = data_Xamin[data_Xamin['Ntot'] > NOMBRE_PHOTONS_NOUVELLE_BORNE]
        coords_photons = SkyCoord(sources_photons["PNT_RA"], sources_photons["PNT_DEC"], unit='deg')
        plot_sources_arcmin(coords_photons, '+', 'limegreen', f"Xamin > {NOMBRE_PHOTONS_NOUVELLE_BORNE}ph", cross=True)

    if show_SExtractor:
        SEX_file = f"Tile-{ra_tile}-{dec_tile}_m12pn_b2_SEXtra_cat.fits"
        SEX_path = os.path.join(os.path.expanduser(catalog_path_SExtractor), SEX_file)
        data_SEX = Table.read(SEX_path)
        coords_SEX = SkyCoord(data_SEX["xpeak_world"], data_SEX["ypeak_world"], unit='deg')
        plot_sources_arcmin(coords_SEX, '^', '#E75480', "SExtractor")

    # --- Mise en forme ---
    ax.set_xlabel("ΔRA [arcmin]", fontsize=22)
    ax.set_ylabel("ΔDec [arcmin]", fontsize=22)
    ax.set_title(rf"ID {id} - Flux amas > {LIM_FLUX_CLUSTER} $\mathrm{{erg/cm^2/s}}$ - AGN > {LIM_FLUX_AGN} $\mathrm{{erg/cm^2/s}}$",
                 pad=20, fontsize=18)
    ax.legend(fontsize=15, facecolor='white', framealpha=0.3, 
                  bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xlim(-taille_image, taille_image)
    ax.set_ylim(-taille_image, taille_image)
    plt.tick_params(axis='both', which='major', labelsize=18)
    
    # --- Sauvegarde ---
    plot_name = f"ID_{id}_7arcmin{'_mr' if mr else ''}.png"
    plot_path = os.path.join(output_dir, plot_name)
    plt.tight_layout()
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Plot saved to: {plot_path}")


In [23]:
List_id_of_clusters = list_ID_Xamin_AMAS

if(True):
    for id in List_id_of_clusters[:5]:
        plot_tverif(id=id , mr = True, show_Xamin_sources = True, show_SExtractor = False, NOMBRE_PHOTONS_NOUVELLE_BORNE = 200,
                    output_dir=f"/local/home/sh275430/Documents/TransformerSurFenestrons/results/{name_dir}/Overlays/Clusters")

Plot saved to: /local/home/sh275430/Documents/TransformerSurFenestrons/results/Simulation1/Overlays/Clusters/Wavelets_images/ID_0_7arcmin_mr.png
Plot saved to: /local/home/sh275430/Documents/TransformerSurFenestrons/results/Simulation1/Overlays/Clusters/Wavelets_images/ID_38_7arcmin_mr.png
Plot saved to: /local/home/sh275430/Documents/TransformerSurFenestrons/results/Simulation1/Overlays/Clusters/Wavelets_images/ID_135_7arcmin_mr.png
Plot saved to: /local/home/sh275430/Documents/TransformerSurFenestrons/results/Simulation1/Overlays/Clusters/Wavelets_images/ID_170_7arcmin_mr.png
Plot saved to: /local/home/sh275430/Documents/TransformerSurFenestrons/results/Simulation1/Overlays/Clusters/Wavelets_images/ID_174_7arcmin_mr.png


In [24]:
missing_indices = np.where(~np.isin(data_Xamin['ID_Xamin'], np.union1d(list_ID_Xamin_AMAS, list_ID_Xamin_AGN)))[0]
missing_ids = data_Xamin['ID_Xamin'][missing_indices]

In [25]:
if(True):
    for id in missing_ids[:5]:
        plot_tverif(id=id , mr = True, show_Xamin_sources = True, show_SExtractor = False, NOMBRE_PHOTONS_NOUVELLE_BORNE = 200,
                    output_dir=f"/local/home/sh275430/Documents/TransformerSurFenestrons/results/{name_dir}/Overlays/Spurious")

Plot saved to: /local/home/sh275430/Documents/TransformerSurFenestrons/results/Simulation1/Overlays/Spurious/Wavelets_images/ID_1126_7arcmin_mr.png
Plot saved to: /local/home/sh275430/Documents/TransformerSurFenestrons/results/Simulation1/Overlays/Spurious/Wavelets_images/ID_1298_7arcmin_mr.png
Plot saved to: /local/home/sh275430/Documents/TransformerSurFenestrons/results/Simulation1/Overlays/Spurious/Wavelets_images/ID_3074_7arcmin_mr.png
Plot saved to: /local/home/sh275430/Documents/TransformerSurFenestrons/results/Simulation1/Overlays/Spurious/Wavelets_images/ID_3359_7arcmin_mr.png
Plot saved to: /local/home/sh275430/Documents/TransformerSurFenestrons/results/Simulation1/Overlays/Spurious/Wavelets_images/ID_4199_7arcmin_mr.png
